# Chapter 2: LLM API Basics

Companion notebook for Chapter 2. Each section corresponds to a listing in the book.

Run the code cells from top to bottom in a fresh kernel after completing the
README setup and this chapter's API prerequisites.


## Setup (Listing 2.1)

In [1]:
from pathlib import Path
from dotenv import load_dotenv

# Locate this checkout, including when Jupyter starts in a chapter folder.
PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "agents").is_dir()
)
import sys
# Use this checkout even if the selected kernel was installed elsewhere.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

%load_ext autoreload
%autoreload 2


In [2]:
cheap_open_ai_model = "gpt-5-nano"
balanced_open_ai_model = "gpt-6-luna"
gemini_model = "gemini/gemini-3.5-flash-lite"

## OpenAI API (Listing 2.2)

In [3]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model=cheap_open_ai_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

print(response.choices[0].message.content)

Paris


## Anthropic API (Listing 2.3)

In [ ]:
# Skipped: no Anthropic API key available.
# from anthropic import Anthropic
#
# client = Anthropic()
#
# response = client.messages.create(
#     model="claude-sonnet-4-6",
#     max_tokens=1024,
#     messages=[
#         {"role": "user", "content": "What is the capital of France?"}
#     ]
# )
#
# print(response.content[0].text)

## Unified API with LiteLLM (Listing 2.4)

In [4]:
from litellm import completion

response = completion(
    model=cheap_open_ai_model,
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response.choices[0].message.content)

response = completion(
    model=gemini_model,
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response.choices[0].message.content)

Hello! How can I help today? I can explain concepts, answer questions, help with writing or editing, brainstorm ideas, plan projects, or assist with code. Tell me a topic or task, or try one of these:

- Explain a concept in simple terms
- Draft or polish an email or document
- Brainstorm ideas for a project or story
- Help with math, science, or coding problems
- Plan a trip or event

What would you like to do?
Hello! How can I help you today?


## Demonstrating API statelessness (Listing 2.5)

In [5]:
from litellm import completion

# First call
response1 = completion(
    model=cheap_open_ai_model,
    messages=[{"role": "user", "content": "My name is Jungjun."}]
)
print(response1.choices[0].message.content)

# Second call
response2 = completion(
    model=cheap_open_ai_model,
    messages=[{"role": "user", "content": "What is my name?"}]
)
print(response2.choices[0].message.content)

Nice to meet you, Jungjun! How can I help you today? If you’d like, tell me your goals or topics you’re interested in. ICan assist with English practice, translations, writing help, coding, studying tips, planning, or just a casual chat. Also, would you like me to use any pronouns or a preferred way to address you?
I don’t know your name yet. I don’t have access to your personal details unless you share them in this chat.

If you tell me your name, I’ll use it while we’re chatting. I won’t remember it after this session unless the platform provides memory features, but you can remind me anytime.

What would you like me to call you?


## Managing conversation history (Listing 2.6)

In [6]:
from litellm import completion

messages = []

# First exchange
messages.append({"role": "user", "content": "My name is Jungjun."})
response1 = completion(model=cheap_open_ai_model, messages=messages)
assistant_message1 = response1.choices[0].message.content
messages.append({"role": "assistant", "content": assistant_message1})
print(assistant_message1)

# Second exchange - includes previous conversation history
messages.append({"role": "user", "content": "What is my name?"})
response2 = completion(model=cheap_open_ai_model, messages=messages)
assistant_message2 = response2.choices[0].message.content
print(assistant_message2)

Nice to meet you, Jungjun! How can I help today? We can do any of the following, or something else you have in mind:

- Practice English (or Korean) conversation
- Proofread or edit writing
- Explain a concept or plan a study/work project
- Brainstorm ideas or solutions
- Translate between Korean and English
- Help with resumes, emails, letters, etc.

Tell me what you'd like to do, and whether you prefer English or Korean.
Your name is Jungjun. Would you like me to call you Jungjun, or do you prefer another name?


## Structured output with Pydantic (Listing 2.7)

In [7]:
from pydantic import BaseModel
from litellm import completion

class ExtractedInfo(BaseModel):
    name: str
    email: str
    phone: str | None = None

response = completion(
    model=cheap_open_ai_model,
    messages=[{
        "role": "user",
        "content": "My name is John Smith, my email is john@example.com, and my phone is 555-1234."
    }],
    response_format=ExtractedInfo
)

result = response.choices[0].message.content
print(result)

{"name":"John Smith","email":"john@example.com","phone":"555-1234"}


## Asynchronous LLM calls (Listing 2.8)

In [8]:
import asyncio
from litellm import acompletion

async def get_response(prompt: str) -> str:
    response = await acompletion(
        model=cheap_open_ai_model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

prompts = [
    "What is 2 + 2?",
    "What is the capital of Japan?",
    "Who wrote Romeo and Juliet?"
]

# Execute all requests concurrently
tasks = [get_response(p) for p in prompts]
results = await asyncio.gather(*tasks)

for prompt, result in zip(prompts, results):
    print(f"Q: {prompt}")
    print(f"A: {result}\n")

Q: What is 2 + 2?
A: 4

Q: What is the capital of Japan?
A: The capital of Japan is Tokyo.

Q: Who wrote Romeo and Juliet?
A: William Shakespeare. It’s a tragedy written by him, likely in the 1590s. If you want, I can share a brief summary or its historical context.



## Rate-limited concurrent requests (Listing 2.9)

In [9]:
import asyncio
from litellm import acompletion

# Limit to 10 concurrent requests
semaphore = asyncio.Semaphore(10)

async def call_llm(prompt: str) -> str:
    """LLM call with rate limiting and automatic retry."""
    async with semaphore:
        response = await acompletion(
            model=cheap_open_ai_model,
            messages=[{"role": "user", "content": prompt}],
            num_retries=3  # Automatic retry with exponential backoff
        )
        return response.choices[0].message.content

# Even with 100 concurrent tasks, only 10 API calls run at a time
prompts = [f"What is {i} + {i}?" for i in range(100)]
tasks = [call_llm(p) for p in prompts]
results = await asyncio.gather(*tasks, return_exceptions=True)

successes = sum(1 for r in results if not isinstance(r, Exception))
print(f"Successful: {successes}/{len(results)}")

Successful: 100/100


## GAIA Benchmark Evaluation

Listings 2.10–2.17 are implemented in `agents.eval.gaia`. The notebook
imports the orchestration function and runs the experiment. Read the source
of `agents/eval/gaia.py` alongside the book listings for the full
implementation.

### Loading GAIA Level 1 problems (Listing 2.10)

In [10]:
# GAIA is gated: accept access on Hugging Face and set HF_TOKEN first.
from datasets import load_dataset

level1_problems = load_dataset("gaia-benchmark/GAIA", "2023_level1", split="validation")
print(f"Number of Level 1 problems: {len(level1_problems)}")

/Users/callummullan-young/Projects/ai-agent-from-scratch/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of Level 1 problems: 53


### Model selection and execution (Listing 2.17)

In [11]:
from agents.eval.gaia import run_experiment

MODELS = [cheap_open_ai_model, balanced_open_ai_model, gemini_model]

subset = level1_problems.select(range(20))
results = await run_experiment(subset, MODELS)

# Per-model accuracy
for model, model_results in results.items():
    correct = sum(1 for r in model_results if r["correct"])
    solvable = sum(1 for r in model_results if r.get("is_solvable"))
    total = len(model_results)
    print(f"{model}: {correct}/{total} correct, {solvable}/{total} judged solvable")

 22%|██▏       | 13/60 [00:04<00:12,  3.68it/s]


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.



 30%|███       | 18/60 [00:05<00:09,  4.49it/s]


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.



 33%|███▎      | 20/60 [00:05<00:07,  5.40it/s]


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.



 45%|████▌     | 27/60 [00:06<00:03, 10.31it/s]


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.



100%|██████████| 60/60 [01:12<00:00,  1.20s/it]

gpt-5-nano: 3/20 correct, 6/20 judged solvable
gpt-6-luna: 7/20 correct, 13/20 judged solvable
gemini/gemini-3.5-flash-lite: 1/20 correct, 8/20 judged solvable
